# Building an Educational Credit Score

---

Author: José Reyes

Project: Credit Risk Fundamentals with NumPy

Estimated Reading Time:
35 minutes

Difficulty:
🟢 Beginner

Business Domain:
Consumer Lending
Credit Risk
Retail Banking

Prerequisites:
- Notebook 01
- Basic Python
- Pandas
- NumPy

---

## 1. Introduction


In the previous notebook we learned which financial variables banks commonly analyze during a credit application.

However, evaluating every variable individually becomes increasingly difficult as the number of applications grows.

Instead of reviewing multiple indicators separately, financial institutions summarize customer financial behavior into a single numerical value known as a Credit Score.

In this notebook we will build an educational weighted scoring model inspired by the factors commonly considered by the Mexican Credit Bureau.

Rather than reproducing a proprietary algorithm, our objective is to understand the business logic behind credit scoring.

## 2. Learning Objectives

By the end of this notebook you will be able to

- Explain why weighted scoring models are used.
- Build a simple scorecard.
- Calculate a weighted credit score.
- Normalize the score to a familiar scale.
- Interpret customer creditworthiness.

## 3. Business Context

Imagine you are working as a Credit Risk Analyst.

Two customers request exactly the same loan.

Loan Amount: $300,000
- Same loan.
- Same product.
- Same term.

Should the bank approve both applications?

Probably not.

The difference lies in the customer's financial behavior.

That behavior must be transformed into measurable indicators.

## 4. Financial Theory

Every scoring model follows the same business logic.

1. Customer Information
2. Risk Factors
3. Weighted Score
5. Credit Score
6. Risk Category
7. Business Decision

## 5. Why Do Banks Use Weighted Scores?

Not every financial behavior has the same predictive power.

For example...

missing several payments is generally much more informative than submitting one additional credit application.


For that reason, scoring models assign different importance (weights) to each factor.

This approach produces a standardized and objective measure of customer risk.

## 6. Educational Scoring Factors

This notebook uses five educational variables inspired by publicly available information regarding the Mexican Credit Bureau.

In [1]:
import pandas as pd

risk_factors = pd.DataFrame({

    "Risk Factor":["Payment History", "Credit Utilization", "Credit History Length", "Credit Mix", "Recent Credit Activity"],
    "Weight (%)":[35, 30, 15, 10, 10],
    "Business Purpose":[
        "Measures repayment behavior.",
        "Measures debt utilization.",
        "Measures financial experience.",
        "Measures product diversification.",
        "Measures recent borrowing activity."
    ]
})

In [2]:
risk_factors

,Risk Factor,Weight (%),Business Purpose
0,Payment History,35,Measures repayment behavior.
1,Credit Utilization,30,Measures debt utilization.
2,Credit History Length,15,Measures financial experience.
3,Credit Mix,10,Measures product diversification.
4,Recent Credit Activity,10,Measures recent borrowing activity.


## 7. Building the Educational Scorecard

Each variable will first be converted into a standardized score between 0 and 100.

Higher values represent healthier financial behavior.

Lower values indicate higher financial risk.

### Payment History

| Late Payments | Score |
| :--- | ---: |
| 0 | 100 |
| 1 | 90 |
| 2 | 75 |
| 3 | 55
| 4+ | 20 |

### Credit Utilization

| Utilization | Score |
| :--- | ---: |
| <30% | 100 |
| 30–50% | 80 |
| 50–70% | 60 |
| >70% | 25 |

### Credit History

| Years | Score |
| :--- | ---: |
| >10 | 100 |
| 5–10 | 80 |
| 2–5 | 60 |
| <2 | 30 |

### Credit Mix

| Products | Score |
| :--- | ---: |
| 4 | 100 |
| 3 | 80 |
| 2 | 60 |
| 1 | 30 |

### Recent Credit Activity

| Inquiries | Score |
| :--- | ---: |
| 0 | 100 |
| 1 | 90 |
| 2 | 75 |
| 3 | 60 |
| 4+ | 30 |

## 8. Mathematical Model

Once each variable has been transformed into an individual score, the overall score is calculated as a weighted average.

The **Raw Credit Score** is calculated as a weighted sum of the five educational risk factors.


$\text{Raw Score}=0.35(\text{Payment History})+0.30(\text{Credit Utilization})+0.15(\text{Credit History Length})+0.10(\text{Credit Mix})+0.10(\text{Recent Credit Activity})$

Where:

| Variable | Meaning |
| :--- | ---: |
| PH | Payment History |
| CU | Credit Utilization |
| CH | Credit History |
| CM | Credit Mix |
| RA | Recent Activity |

Unlike a simple arithmetic mean, a weighted score recognizes that not all financial behaviors contribute equally to credit risk.

For example, a history of missed payments is generally more predictive of future default than having submitted one additional credit inquiry.

The weights assigned to each factor reflect their relative importance within our educational scoring model.

Now, lets create the customers dataframe

In [3]:
customers = pd.DataFrame({
    "Customer":["Carlos", "Andrea"],
    "Late_Payments":[0, 4],
    "Credit_Utilization":[22, 91],
    "Credit_History_Years":[11, 2],
    "Credit_Mix":[3, 1],
    "Recent_Inquiries":[0, 5]
})

In [4]:
customers

,Customer,Late_Payments,Credit_Utilization,Credit_History_Years,Credit_Mix,Recent_Inquiries
0,Carlos,0,22,11,3,0
1,Andrea,4,91,2,1,5


### Business Question

Without writing any code yet,

How would you assign a score to each customer?

Would every variable contribute equally?

Should missing one payment have the same impact as using 90% of the available credit?

For this project, all scoring functions will be implemented inside the **src** package.

This approach provides several advantages:

- Reusable business rules
- Cleaner notebooks
- Easier testing
- Better software organization
- Separation between business logic and analysis

Our first scoring function will evaluate the customer's payment history.

In [5]:
# Rooting the src path
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [6]:
# importing modules in src
from src.credit_score import (
    payment_history_score,
    utilization_score,
    credit_history_score,
    credit_mix_score,
    recent_activity_score,
    calculate_raw_score,
    normalize_credit_score,
    assign_credit_rating,
)

In [7]:
# Applying functions to each variable
customers["PH"] = (customers["Late_Payments"].apply(payment_history_score))
customers["CU"] = (customers["Credit_Utilization"].apply(utilization_score))
customers["CH"] = (customers["Credit_History_Years"].apply(credit_history_score))
customers["CM"] = (customers["Credit_Mix"].apply(credit_mix_score))
customers["RA"] = (customers["Credit_Mix"].apply(recent_activity_score))

# Raw Score
customers["Raw_Score"] = customers.apply(calculate_raw_score,axis=1)


In [8]:
customers

,Customer,Late_Payments,Credit_Utilization,Credit_History_Years,Credit_Mix,Recent_Inquiries,PH,CU,CH,CM,RA,Raw_Score
0,Carlos,0,22,11,3,0,100,100,100,80,60,98.0
1,Andrea,4,91,2,1,5,20,25,60,40,90,30.5


### Business Interpretation


The educational raw score summarizes five independent dimensions of customer financial behavior into a single weighted metric.

Rather than analyzing each variable separately, the scoring model provides an objective indicator of overall creditworthiness.

At this stage, the score ranges from **0 to 100**. In the next section, we will transform this value into a familiar **400–850 educational credit score**, similar to the scale commonly used by credit bureaus.

## 9. Normalizing the Raw Score


The weighted score calculated in the previous section ranges from **0 to 100**.

Although useful for internal calculations, most financial institutions communicate creditworthiness using a standardized score.

For educational purposes, we will map our raw score onto a **400–850** scale inspired by the Mexican Credit Bureau.

This transformation preserves the relative ranking of customers while making the score easier to interpret.

### Mathematical Model

$\text{Credit Score}= 400 + 4.5 \times \text{Raw Score}$

In [9]:
# Applying the function to normalize
customers["Credit_Score"] = (customers["Raw_Score"].apply(normalize_credit_score))

In [10]:
customers[["Customer", "Raw_Score", "Credit_Score"]]

,Customer,Raw_Score,Credit_Score
0,Carlos,98.0,841
1,Andrea,30.5,537


### Business Interpretation

The educational credit score now resembles the scale commonly used by financial institutions.

Although the values do not represent the official Buró de Crédito methodology, they provide an intuitive interpretation of customer creditworthiness.

Higher scores indicate healthier financial behavior and lower expected credit risk.

## 10. Risk Categories

| Credit Score | Category |
| :--- | ---: |
| 750–850 | Excelent |
| 650–749 | Good |
| 550–649 | Fair |
| 400–549 | Poor |

In [11]:
customers["Credit_Rating"] = (customers["Credit_Score"].apply(assign_credit_rating))

In [16]:
customers[["Customer", "Raw_Score", "Credit_Score", "Credit_Rating"]]

,Customer,Raw_Score,Credit_Score,Credit_Rating
0,Carlos,98.0,841,Excellent
1,Andrea,30.5,537,Poor


# What's Next?

In the next notebook we will move beyond scoring.

The educational credit score developed here will become an input for business decisions such as:

- Loan approval or rejection
- Credit limit assignment
- Interest rate pricing

These decision rules will later power our interactive Credit Approval Simulator.